# Oferta Hídrica Subterránea — Análisis Estadístico Ambiental

> **Contexto de dominio:** [`docs/fuentes/oferta_hidrica.md`](../../docs/fuentes/oferta_hidrica.md)
> **Bloque:** A — Gestión | **Línea:** `oferta_hidrica`
> **Variable principal:** `caudal` (m³/s) / Nivel Estático (m) / Caudal de Extracción (L/s)
> **Modelos sugeridos:** SARIMA, SARIMAX, PROPHET, XGBOOST, RANDOM_FOREST
> **Flujo:** `Plan.md` sección 3 — ciclo estadístico completo.

---

## ¿Qué es la oferta hídrica subterránea en Colombia?

El agua subterránea constituye una de las fuentes estratégicas más críticas para el abastecimiento humano y la sostenibilidad de los ecosistemas colombianos. Abastece a una fracción fundamental de la población rural y periurbana, y sostiene el caudal base de los ríos durante épocas de estiaje, cuando las fuentes superficiales disminuyen drásticamente. A pesar de su importancia, enfrenta presiones crecientes de agotamiento y contaminación, derivadas de la extracción no regulada, la deficiente planificación urbana y los efectos del cambio climático sobre las zonas de recarga.

En Colombia, la gestión de la oferta hídrica subterránea se articula a través de dos instrumentos principales:

- **POMCA** (Plan de Ordenación y Manejo de Cuencas Hidrográficas): incluye la caracterización hidrogeológica de la cuenca.
- **PMAA** (Plan de Manejo Ambiental de Acuíferos): instrumento específico para la regulación y protección de los sistemas de agua subterránea.

## Instituciones responsables

| Institución | Rol |
|---|---|
| **MADS** | Dicta la Política Nacional (PNGIRH) y lineamientos normativos |
| **SGC** (Serv. Geológico Colombiano) | Exploración geológica, evaluación de reservas, investigación del subsuelo |
| **IDEAM** | Monitoreo nacional (ENA), protocolos técnicos, administración del SIRH |
| **CARs** (ej. CORPOCESAR, CVC) | Expiden concesiones, ejecutan PMAA y POMCA, aplican tasas retributivas |
| **Empresas / Usuarios** | Sector agrícola e industrial que extrae el recurso y aporta datos operativos |

## Preguntas que responde este análisis

1. ¿Cuál es la tendencia en los niveles piezométricos o caudales extraídos? ¿Existe señal de sobreexplotación?
2. ¿El Índice de Uso del Agua (IUA) supera los umbrales de presión crítica (>100%)?
3. ¿Existe una correlación temporal con el fenómeno ENSO? ¿Con qué rezago responde el acuífero?
4. ¿Qué modelo estadístico predice mejor el comportamiento del caudal a 6 meses horizonte?
5. ¿Cuáles son las épocas del año con mayor riesgo de desabastecimiento?

---

**Antes de ejecutar:** Leer `docs/fuentes/oferta_hidrica.md` para familiarizarse con los rangos físicos de las variables, la normativa aplicable y las fuentes de datos oficiales.

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "oferta_hidrica"
VARIABLE = "caudal"
UNIDAD = "m³/s"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio

Esta celda carga la ficha técnica de la línea `oferta_hidrica` directamente desde `docs/fuentes/`. La ficha contiene el resumen ejecutivo del dominio, las variables ambientales con sus rangos físicos, la normativa colombiana aplicable y las preguntas analíticas más relevantes.

Se recomienda revisar especialmente:
- La tabla de **Variables ambientales clave**: rangos físicos esperados para el nivel estático (3.3 a 30 m), conductividad (~100 a 1000 µS/cm) y caudal de extracción (1 a 50+ L/s).
- La sección de **Indicadores**: IUA, Índice de Escasez de Agua Subterránea, índices GOD/DRASTIC de vulnerabilidad.
- Las **fuentes de datos**: inventarios FUNIAS, expedientes de concesión de CARs, bases de datos SIRH.

> **Nota sobre outliers (ADR-002):** Los picos de caudal de extracción o los abatimientos bruscos de nivel son señal real del comportamiento del acuífero bajo bombeo. No aplicar clipping automático. El módulo `preprocessing/outliers.py` es opt-in explícito.

In [ ]:
ficha = DOCS_FUENTES / "oferta_hidrica.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### ¿Qué datos necesito?

Para el análisis de oferta hídrica subterránea se requiere una **serie temporal** con al menos las siguientes columnas:

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | datetime | Fecha de la medición (mensual o diaria) |
| `caudal` | float | Caudal extraído (m³/s) o Nivel Estático (m) |
| `estacion` | str (opcional) | Código o nombre del pozo/piezómetro |
| `conductividad` | float (opcional) | µS/cm — indicador de calidad |

### Fuentes oficiales de datos en Colombia

- **IDEAM — SIRH:** [sirh.ideam.gov.co](http://sirh.ideam.gov.co) — Registros de concesiones, niveles y calidad.
- **IDEAM — DHIME:** [dhime.ideam.gov.co](http://dhime.ideam.gov.co) — Descarga de series hidrometeorológicas.
- **CARs regionales:** Cada CAR mantiene sus propias bases de datos de monitoreo piezométrico. Ej. CORPOCESAR, CVC, CORTOLIMA.
- **SGC — FUNIAS:** Inventario Nacional de Puntos de Agua Subterránea.

### Frecuencia y horizonte recomendados

- **Frecuencia mínima:** Mensual. Semestral para piezometría manual.
- **Horizonte mínimo:** 10 años para análisis de tendencia. 20+ años para análisis climático con ENSO.

> **Nota:** El dataset sintético de ejemplo usa `pd.date_range` con frecuencia mensual (`freq="ME"`) y una distribución Gamma para simular caudales positivos con variabilidad realista. Para datos reales, reemplazar la sección comentada con `load_csv()`.

In [ ]:
# df = load_csv("data/raw/oferta_hidrica.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "caudal": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 1b. Hidrogeologia: Piezometria, transmisividad y curva de abatimiento

El inventario FUNIAS levanta los parametros clave del acuifero por pozo:

| Parametro | Simbolo | Unidad | Rango tipico |
|---|---|---|---|
| Nivel estatico | Ne | m | 3 - 30 m |
| Nivel dinamico | Nd | m | > Ne |
| Transmisividad | T | m2/dia | 10 - 5000 m2/dia |
| Caudal extraccion | Q | L/s | 1 - 50+ L/s |

**Ecuacion de Theis** (bombeo a largo plazo, sin recarga):
```
s(r,t) = Q/(4*pi*T) * W(u)       u = r^2*S/(4*T*t)
```
Donde W(u) = funcion de pozo (well function), r = distancia al pozo (m), S = coeficiente de almacenamiento (adim), t = tiempo (dias).

**Kriging Empirico Bayesiano (EBK):** mejor opcion para mapas de isopiezas porque estima la varianza del modelo variografico en lugar de asumirla fija.

**MODFLOW** (USGS): modelo de diferencias finitas para simular el acuifero en 3D; se conecta a Python mediante `flopy`.

In [ ]:
# Inventario FUNIAS sintetico + curva de abatimiento (Theis) + Kriging
np.random.seed(33)
n_pozos = 25

# Inventario FUNIAS (pozos del acuifero)
df_pozos = pd.DataFrame({
    'pozo_id': [f'PZ-{i+1:03d}' for i in range(n_pozos)],
    'ne_m': np.random.uniform(3, 25, n_pozos).round(1),        # nivel estatico
    'q_ls': np.random.uniform(1, 40, n_pozos).round(1),         # caudal extraccion
    'transmisiv': np.random.lognormal(4, 1, n_pozos).round(1),  # m2/dia
    'lat': np.random.uniform(4.5, 6.5, n_pozos),                # lat Colombia
    'lon': np.random.uniform(-75.5, -73.5, n_pozos),            # lon Colombia
})
df_pozos['nivel_piezom'] = df_pozos['ne_m'] + np.random.uniform(50, 200, n_pozos)  # cota

# Curva de abatimiento Theis para pozo representativo
Q_bomba = 10.0    # L/s = 0.01 m3/s
T_trans = 150.0   # m2/dia (transmisividad tipica)
S_almac = 0.001   # coeficiente almacenamiento (acuifero confinado)
r_obs = 50.0      # m distancia pozo de observacion
t_dias = np.logspace(-2, 2, 200)  # 0.01 a 100 dias

# Aproximacion Cooper-Jacob (valida para u < 0.05)
u = r_obs**2 * S_almac / (4 * T_trans * t_dias)
abatim = np.where(u < 0.05,
    (Q_bomba * 86.4) / (4 * np.pi * T_trans) * (-0.5772 - np.log(u)),
    np.nan)

print(f'Inventario FUNIAS: {n_pozos} pozos')
print(f'Transmisividad | mediana={df_pozos["transmisiv"].median():.1f} m2/dia')
print(f'Caudal medio extraccion: {df_pozos["q_ls"].mean():.1f} L/s')
print(f'Abatimiento maximo en r={r_obs}m despues de 100 dias: {np.nanmax(abatim):.2f} m')
print()
print('Para mapa de isopiezas: usar pykrige.OrdinaryKriging o EmpiricalBayesianKriging')
print('Para modelo MODFLOW 3D: flopy.modflow (USGS) con Python/FloPy')
df_pozos[['pozo_id','ne_m','q_ls','transmisiv','nivel_piezom']].head(10)

## 2. Validación y EDA

### ¿Qué hace `validate()` en esta línea?

La función `validate()` aplica rangos físicos específicos para la línea `oferta_hidrica` definidos en `config.py`. Verifica que:

- El caudal no sea negativo (físicamente imposible).
- Los niveles estáticos estén dentro del rango esperado para el acuífero (entre 3.3 y 30+ m según la ficha).
- La conductividad eléctrica no supere rangos de alerta de contaminación (> 1000 µS/cm).
- No haya fechas duplicadas ni columnas de fecha con valores nulos.

### Señales de alerta en la validación

Si `val.summary()` reporta errores, considerar:

- **Valores negativos de caudal:** Posibles errores de digitación o sensor.
- **Saltos bruscos de nivel (> 5 m en un mes):** Puede indicar inicio/fin de prueba de bombeo — registrar en metadata.
- **Periodos sin datos:** Frecuentes en el SIRH por falta de continuidad institucional. La sección 6 aplica imputación.

> **ADR-006:** Usar siempre `validate(df, linea_tematica="oferta_hidrica")` para que las verificaciones sean específicas de dominio, no genéricas. Los rangos físicos difieren entre una cuenca andina y una llanura aluvial.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_oferta_hidrica.html",
       title="EDA — Oferta Hídrica", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

### ¿Qué buscar en las gráficas?

Las visualizaciones de esta sección permiten una primera lectura del comportamiento del caudal antes de aplicar cualquier estadística formal. En el contexto de la oferta hídrica, se deben identificar:

**En la serie temporal (plot_series):**
- Tendencia general: ¿hay una disminución progresiva del caudal? Podría indicar sobreexplotación del acuífero o reducción de zonas de recarga por cambio en uso del suelo.
- Pulsos estacionales: ¿el caudal responde a la temporada de lluvias? La sincronía con el régimen pluviométrico indica conexión hidráulica activa.
- Eventos extremos: picos o caídas abruptas asociadas a fenómenos ENSO o a pruebas de bombeo.

**En las medias estacionales (plot_seasonal_means):**
- Meses de máxima extracción vs. meses de máxima recarga: si la extracción supera la recarga estacional, el acuífero no se recupera.
- Bimodalidad: en la zona andina colombiana, los meses de recarga suelen ser marzo-mayo y septiembre-noviembre.
- Meses de mayor vulnerabilidad: diciembre-febrero y junio-agosto (veranillos) son períodos críticos de baja recarga y alta demanda agrícola.

In [ ]:
plot_series(df, "fecha", "caudal", title="Oferta Hídrica — caudal (m³/s)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "caudal", period="month")
plt.show()

## 3c. Mapa de isopiezas y curva de abatimiento — Kriging + Theis

El mapa de isopiezas (lineas de igual carga hidraulica) muestra la direccion del flujo subterraneo y permite identificar zonas de recarga (valores altos) y descarga (valores bajos). Se genera con **Kriging Ordinario** o **EBK** sobre los niveles piezometricos de los pozos FUNIAS.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Dispersion de pozos FUNIAS + nivel piezometrico (proxy mapa isopiezas)
sc = ax1.scatter(df_pozos['lon'], df_pozos['lat'],
                 c=df_pozos['nivel_piezom'], cmap='Blues_r', s=60, alpha=0.85, edgecolors='gray')
plt.colorbar(sc, ax=ax1, label='Nivel piezometrico (m.s.n.m.)')
ax1.set_xlabel('Longitud'); ax1.set_ylabel('Latitud')
ax1.set_title('Pozos FUNIAS — Mapa de isopiezas\n(Kriging EBK con pykrige)', fontweight='bold')
ax1.grid(alpha=0.3)
# Anotar pozos con mayor transmisividad
top3 = df_pozos.nlargest(3, 'transmisiv')
for _, row in top3.iterrows():
    ax1.annotate(f"{row['pozo_id']}\nT={row['transmisiv']:.0f}",
                 (row['lon'], row['lat']), fontsize=6, color='darkblue')

# Panel B: Curva de abatimiento Theis / Cooper-Jacob
ax2.semilogx(t_dias, abatim, lw=2, color='#e74c3c', label=f'Abatimiento r={r_obs}m')
ax2.axhline(5.0, color='orange', ls='--', lw=1.5, label='Umbral abatim. 5m (alerta)')
ax2.axhline(10.0, color='red', ls='--', lw=1.5, label='Umbral abatim. 10m (critico)')
ax2.set_xlabel('Tiempo de bombeo (dias, escala log)')
ax2.set_ylabel('Abatimiento (m)')
ax2.set_title(f'Curva abatimiento Theis/Cooper-Jacob\nQ={Q_bomba} L/s, T={T_trans} m2/dia', fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.suptitle('Hidrogeologia — FUNIAS + Theis + Kriging (MODFLOW/FloPy para modelo 3D)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

t_alerta = t_dias[np.nanargmax(abatim >= 5.0)] if (abatim >= 5.0).any() else None
if t_alerta:
    print(f'Abatimiento supera 5m despues de {t_alerta:.1f} dias de bombeo')

## 3b. Covariable ENSO (ONI) — Por qué y con qué lag

### ¿Por qué incorporar ENSO en el análisis de oferta hídrica?

El fenómeno ENSO (El Niño-Oscilación del Sur) es el principal modulador de la variabilidad climática interanual en Colombia. Afecta directamente la precipitación sobre las zonas de recarga de los acuíferos:

- **El Niño (ONI > 0.5):** Reduce la precipitación en la mayor parte del país. Las zonas de recarga reciben menos lluvia, el nivel estático de los pozos desciende, y la demanda de agua subterránea aumenta por la escasez superficial. Resultado: mayor presión sobre el acuífero en el peor momento.
- **La Niña (ONI < -0.5):** Aumenta la precipitación. Las zonas de recarga se saturan, los niveles freáticos suben y los ríos incrementan su aporte al acuífero (recarga inducida). Riesgo de inundación de pozos en zonas de nivel freático alto.

### ¿Por qué usar lag = 4 meses? (ADR-007)

La respuesta de los acuíferos a la precipitación no es instantánea. El agua de lluvia debe infiltrarse, percolarse a través de la zona no saturada y llegar a la zona saturada antes de que el nivel freático responda. Este tiempo de tránsito depende de la profundidad del nivel freático, la permeabilidad del suelo y la geometría del acuífero.

Para la línea `oferta_hidrica`, la literatura colombiana y el análisis de correlación cruzada entre el ONI y los registros piezométricos sugieren un **lag de 4 meses**: el impacto de un evento ENSO en el nivel estático del acuífero se manifiesta aproximadamente 4 meses después del pico del ONI. Este valor está configurado en `config.ENSO_LAG_MESES["oferta_hidrica"]`.

> **ADR-007:** Siempre usar `enso_lagged()` en lugar de `enso_dummy()`. La función `enso_lagged` aplica automáticamente el lag configurado por línea temática, evitando el error frecuente de correlacionar el ONI contemporáneo con variables hidrológicas que responden con desfase.

### Interpretación de las columnas ENSO resultantes

Después de ejecutar `enso_lagged()`, se agregarán columnas al DataFrame:
- `oni_lag4`: Índice ONI rezagado 4 meses.
- `enso_fase`: Categoría del evento (niño / niña / neutro) según `config.ENSO_THRESHOLDS`.

In [ ]:
# --- Covariable ENSO (lag específico para Oferta Hídrica) ---
oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica="oferta_hidrica")
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial — Estacionariedad y tendencia

### ¿Por qué verificar estacionariedad? (ADR-004)

Antes de ajustar cualquier modelo ARIMA o SARIMA, es **obligatorio** verificar si la serie temporal de caudal es estacionaria. Una serie no estacionaria tiene media y/o varianza que cambian en el tiempo, lo que viola los supuestos de los modelos de series temporales clásicos y genera pronósticos no confiables.

La ADR-004 establece el uso **conjunto** de dos pruebas complementarias:

| Prueba | Hipótesis nula (H₀) | Conclusión si se rechaza H₀ |
|---|---|---|
| **ADF** (Augmented Dickey-Fuller) | La serie tiene raíz unitaria (no estacionaria) | La serie es estacionaria |
| **KPSS** (Kwiatkowski-Phillips-Schmidt-Shin) | La serie es estacionaria | La serie tiene raíz unitaria |

**Interpretación conjunta:**
- ADF rechaza H₀ **y** KPSS no rechaza H₀ → Serie estacionaria. ARIMA con d=0.
- ADF no rechaza H₀ **y** KPSS rechaza H₀ → Serie claramente no estacionaria. Aplicar diferenciación (d=1).
- Ambas concuerdan en no estacionariedad → Diferenciar y repetir el test.
- Resultados contradictorios → Posible cambio estructural. Evaluar partición de la serie o modelos de ruptura.

### Interpretación en el contexto de oferta hídrica

Para series de caudal de acuíferos:
- Una **tendencia negativa estacionaria** (caudal disminuye pero de forma estable) puede indicar sobreexplotación progresiva.
- Una **raíz unitaria** (paseo aleatorio) sugiere que el caudal futuro depende enteramente del pasado sin tendencia determinista — útil para modelos ARIMA con d=1.
- Una **estacionalidad fuerte** (patrón anual claro) requiere modelos SARIMA con la componente estacional P, D, Q, 12.

### ¿Qué significa el resultado de Mann-Kendall?

La prueba Mann-Kendall es no paramétrica y detecta tendencias monótonas sin asumir distribución normal:
- `trend = "increasing"` con p < 0.05 → Tendencia ascendente significativa del caudal.
- `trend = "decreasing"` con p < 0.05 → **Alerta:** El caudal disminuye consistentemente — posible señal de agotamiento del acuífero.
- `slope` (Sen's slope): cantidad de cambio por unidad de tiempo. Por ejemplo, -0.02 m³/s/año indica reducción de 0.02 m³/s cada año.

> **ADR-004:** No omitir la prueba de estacionariedad. Aplicar ARIMA sobre una serie no estacionaria sin diferenciar produce residuos correlacionados y pronósticos que no convergen.

In [ ]:
ts = df.set_index("fecha")["caudal"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas — IUA y normas colombianas

### ¿Qué normas aplican a la oferta hídrica subterránea?

Para el caudal de extracción y el nivel del agua subterránea, los umbrales normativos no son límites de calidad sino **umbrales de gestión** definidos por el IDEAM en el Estudio Nacional del Agua (ENA):

| Indicador | Umbral | Categoría | Norma de referencia |
|---|---|---|---|
| **IUA** (Demanda/Oferta × 100) | 0 – 20% | Bajo — sin restricción | IDEAM / ENA |
| **IUA** | 20 – 50% | Moderado — monitoreo | IDEAM / ENA |
| **IUA** | 50 – 100% | Alto — restricción de nuevas concesiones | IDEAM / ENA |
| **IUA** | > 100% | Muy alto / Crítico — sobreexplotación | IDEAM / ENA |
| **Índice de Escasez** | > 50% | Escasez alta | Res. 872/2006 MADS |

Para calcular el IUA con los datos disponibles:
```python
# Si se tienen datos de demanda:
iua = (df["demanda"] / df["caudal"]) * 100
meses_criticos = iua[iua > 100].index
```

### ¿Qué hace `exceedance_report()` con la variable `caudal`?

La función `exceedance_report()` busca en `config.py` los umbrales normativos registrados para la variable especificada. Para `caudal`, puede que no existan umbrales directos de calidad (como sí los hay para PM2.5 o OD), sino que el análisis de excedencias se hace vía el IUA calculado externamente.

Si el resultado está vacío, el caudal como variable no tiene norma de calidad directa en Colombia — lo relevante es calcular el IUA como se muestra arriba.

> **ADR-005:** Todos los umbrales normativos colombianos deben centralizarse en `config.py`. No hardcodear valores como 100% (IUA crítico) directamente en el código del notebook — usar `config.IUA_THRESHOLDS`.

In [ ]:
rep = exceedance_report(df["caudal"], variable="caudal")
if rep.empty:
    print("Sin normas colombianas registradas para 'caudal'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["caudal"], method="linear")
print(f"Faltantes antes: {df["caudal"].isna().sum()} | "
      f"después: {df_clean["caudal"].isna().sum()}")

## 7. Modelos predictivos — ¿Por qué NSE y KGE son las métricas primarias?

### El problema con el RMSE en hidrología

El RMSE (Root Mean Square Error) es la métrica de error más usada en machine learning general, pero en hidrología tiene una limitación crítica: **no discrimina entre un modelo que replica bien el caudal medio y uno que replica bien los picos y valles**. Un modelo que siempre predice el promedio histórico puede tener un RMSE aceptable, pero no tiene ningún valor predictivo real.

Por eso, la hidrología adopta métricas específicas:

### NSE — Nash-Sutcliffe Efficiency

```
NSE = 1 - (Σ(Qobs - Qsim)²) / (Σ(Qobs - Qmean)²)
```

El NSE compara el error del modelo contra el error de usar siempre la media como predicción. Un NSE = 0 significa que el modelo es tan bueno como usar la media — no añade valor. Un NSE = 1 es predicción perfecta.

| NSE | Evaluación del modelo |
|---|---|
| > 0.75 | Bueno — modelo confiable para gestión |
| 0.65 – 0.75 | Satisfactorio — uso con precaución |
| 0.50 – 0.65 | Aceptable — para análisis preliminar |
| < 0.50 | Insatisfactorio — no usar para decisiones |

**Limitación del NSE:** es sensible a valores extremos. Un solo caudal pico muy mal predicho puede derrumbar el NSE aunque el modelo sea bueno para caudales medios.

### KGE — Kling-Gupta Efficiency

```
KGE = 1 - √((r-1)² + (α-1)² + (β-1)²)
```

Donde r = correlación de Pearson, α = relación de variabilidades, β = relación de medias. El KGE descompone el error en tres componentes interpretables por separado:
- **r < 0.8:** El modelo no captura la dinámica temporal (picos y valles en los momentos equivocados).
- **α < 0.7 o > 1.3:** El modelo subestima o sobreestima la variabilidad del caudal.
- **β < 0.9 o > 1.1:** El modelo tiene sesgo sistemático en el volumen predicho.

> **ADR-003:** Para caudal (variable siempre positiva), NSE y KGE son las métricas primarias. RMSE puede usarse como métrica secundaria para comparación, pero no como criterio de selección del modelo final. RMSLE no aplica para series que pueden tener valores muy cercanos a cero (caudales de estiaje extremo).

In [ ]:
ts = df_clean.set_index("fecha")["caudal"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones metodológicas

### Síntesis del ciclo estadístico para oferta hídrica

Este notebook implementó el ciclo estadístico completo para la línea `oferta_hidrica`:

1. **Validación de dominio** (`validate()`): verificación de rangos físicos específicos del acuífero.
2. **EDA**: caracterización del régimen de caudales, detección de tendencias visuales y estacionalidad.
3. **ENSO con lag 4 meses** (`enso_lagged()`): incorporación de la variabilidad climática interanual como covariable.
4. **Estacionariedad** (ADF + KPSS): cumplimiento de ADR-004 antes del modelado.
5. **Tendencia** (Mann-Kendall): detección de señales de sobreexplotación o recuperación del acuífero.
6. **Excedencias** (`exceedance_report()`): evaluación del IUA y umbrales de escasez.
7. **Modelos** (SARIMA, SARIMAX, Prophet, XGBoost, RandomForest): comparación con métricas NSE, KGE.

### Decisiones metodológicas clave

| Decisión | Regla aplicada | ADR |
|---|---|---|
| No hacer clipping de outliers | Picos de extracción = señal real de bombeo | ADR-002 |
| ADF + KPSS obligatorios | Verificación de estacionariedad pre-ARIMA | ADR-004 |
| ENSO con lag = 4 meses | Tiempo de respuesta del acuífero al ONI | ADR-007 |
| NSE y KGE como métricas primarias | Estándar hidrológico, no RMSE genérico | ADR-003 |
| Umbrales IUA en config.py | Centralización normativa colombiana | ADR-005 |

### Hallazgos de referencia (datos sintéticos)

Los datos sintéticos de este ejemplo no tienen interpretación ambiental. Al trabajar con datos reales, documentar en esta sección:
- Resultado del test ADF/KPSS y número de diferenciaciones aplicadas.
- Tendencia Mann-Kendall: slope en m³/s/año y significancia estadística.
- Meses con IUA > 50% y > 100%.
- Mejor modelo según NSE y KGE con los parámetros seleccionados.
- Registrar decisiones importantes en `docs/decisiones.md`.

### Normativa y referencias
- `docs/fuentes/oferta_hidrica.md` — Ficha técnica completa con normativa colombiana
- Decreto 1640 de 2012 y Decreto 1076 de 2015 — Marco legal POMCA y PMAA
- IDEAM — Estudio Nacional del Agua (ENA) — Metodología IUA e IRH
- Resolución 872 de 2006 — Índice de escasez de agua subterránea

## 9. Cómo adaptar a datos reales

Esta sección es la guía práctica para quien quiera ejecutar el notebook con datos reales de un proyecto o CAR colombiana.

### Paso 1 — Obtener los datos

**Fuentes recomendadas según el tipo de análisis:**

| Tipo de dato | Fuente | Cómo acceder |
|---|---|---|
| Caudales de extracción (L/s) | SIRH / expedientes de concesión CAR | Solicitud a la CAR correspondiente |
| Niveles piezométricos (m) | Redes de monitoreo piezométrico CAR | Reportes de PMAA o FUNIAS del SGC |
| Series hidrometeorológicas | IDEAM — DHIME | dhime.ideam.gov.co (requiere cuenta) |
| Concesiones vigentes | SIRH — Módulo RURH | sirh.ideam.gov.co |
| Inventario de pozos | SGC — FUNIAS | geoportal.sgc.gov.co |

### Paso 2 — Preparar el archivo CSV

El archivo debe tener como mínimo estas columnas con estos nombres exactos:

```
fecha,caudal
2010-01-01,12.5
2010-02-01,11.8
2010-03-01,15.2
...
```

- `fecha`: formato ISO (`YYYY-MM-DD`). Si los datos son mensuales, usar el primer día del mes.
- `caudal`: caudal extraído en m³/s o nivel estático en metros. Agregar columna para cada variable adicional.
- Guardar en `data/raw/oferta_hidrica.csv`.

### Paso 3 — Activar la carga real

En la celda de la sección 1, descomentar la línea real y comentar los datos sintéticos:

```python
df = load_csv("data/raw/oferta_hidrica.csv", date_col="fecha")
# (comentar las líneas de datos sintéticos)
```

### Paso 4 — Ajustar `LINEA` y `VARIABLE`

Al inicio del notebook (sección 0), verificar que:
```python
LINEA = "oferta_hidrica"   # No cambiar
VARIABLE = "caudal"         # O "nivel_estatico" si se trabaja con piezometría
UNIDAD = "m³/s"             # O "m" para nivel piezométrico
```

### Paso 5 — Interpretar y documentar

Al completar el análisis:
1. Documentar los hallazgos principales en la sección 8 de conclusiones.
2. Registrar las decisiones metodológicas no obvias en `docs/decisiones.md` (ej. por qué se eligió el método de imputación, si se aplicó transformación logarítmica).
3. Si hay outliers que se decidió remover (excepción a ADR-002), documentar la justificación técnica.

### Checklist de verificación antes de entregar

- [ ] Serie validada con `validate()` sin errores críticos.
- [ ] ADF + KPSS ejecutados (ADR-004).
- [ ] ENSO incorporado con `enso_lagged()` (ADR-007).
- [ ] IUA calculado y categorizado según `config.IUA_THRESHOLDS`.
- [ ] Modelo seleccionado con NSE > 0.65 y KGE > 0.65 (ADR-003).
- [ ] Decisiones documentadas en `docs/decisiones.md`.